In [12]:
import pandas as pd
import numpy as np

# Trabalho 1 - Rafael Gabriel 121087555

## Introdução

O trabalho 1 consiste da criação de um classificador para apoio na decisão de aprovação de crédito. Esse trabalho está separado em dois grupos:
- Análise exploratória dos dados;
- Tratamento dos dados;
- Treinamento e validação;
- Predição e envio. 

## Análise exploratória dos dados

Essa etapa consiste da análise dos dados, verificação se há outliers, colunas com valores em branco e etc. O primeiro passo dessa etapa é a importação dos dados para um dataframe do pandas. A coluna ```id_solicitante``` já foi removida pois não traz nenhuma informação.

In [13]:
df = pd.read_csv("data/conjunto_de_treinamento.csv",  na_values=[' ', 'NULL']).drop(columns='id_solicitante')
df.describe().T

,count,mean,std,min,25%,50%,75%,max
produto_solicitado,20000.0,1.284350,1.008239,1.0,1.0,1.0,1.0,7.0
dia_vencimento,20000.0,13.147250,6.748507,1.0,10.0,10.0,20.0,25.0
tipo_endereco,20000.0,1.006350,0.079435,1.0,1.0,1.0,1.0,2.0
idade,20000.0,42.352500,14.930177,7.0,31.0,40.0,52.0,106.0
estado_civil,20000.0,2.120850,1.332004,0.0,1.0,2.0,2.0,7.0
qtde_dependentes,20000.0,0.666400,1.236725,0.0,0.0,0.0,1.0,53.0
grau_instrucao,20000.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
nacionalidade,20000.0,0.961600,0.202305,0.0,1.0,1.0,1.0,2.0
codigo_area_telefone_residencial,16466.0,64.362626,38.701585,1.0,25.0,69.0,100.0,126.0
tipo_residencia,19464.0,1.261303,0.883580,0.0,1.0,1.0,1.0,5.0


A partir do dicionário de dados (dicionario_de_dados.xlsx) é possível identificar que a coluna grau_instrucao possui apenas valores 0. Por precaução, foi procurado mais colunas com possíveis valores iguais.

In [14]:
valoresDistintos = df.nunique(dropna=False).sort_values()

tabelaDistintos = pd.DataFrame({
    "Valores distintos": valoresDistintos,
    "Exemplo de valor": df[valoresDistintos.index].iloc[0],
})

# Drop das colunas que possuem apenas 1 valor distinto
colunasConstantes = tabelaDistintos[tabelaDistintos["Valores distintos"] == 1].index.tolist()
df = df.drop(columns=colunasConstantes)

tabelaDistintos

,Valores distintos,Exemplo de valor
grau_instrucao,1,0
possui_telefone_celular,1,N
possui_telefone_residencial,2,Y
tipo_endereco,2,1
possui_cartao_visa,2,0
possui_cartao_mastercard,2,0
possui_carro,2,1
possui_cartao_diners,2,0
possui_outros_cartoes,2,0
possui_telefone_trabalho,2,N


Pelo resultado, é possível notar que além da coluna ```grau_instrucao``` a coluna ```possui_telefone_celular``` também possui todos os valores iguais a N, por isso foi removida também.

A partir do método ```describe``` também é possível notar que a coluna ```local_onde_reside``` é idêntica a coluna ```local_onde_trabalha```. Além de remover uma delas, já que duas colunas iguais não acrescentam nenhuma informação extra, também foi realizado a procura por mais colunas iguais.

In [15]:
colunas = df.columns.tolist()
colunasIguais = []

for i in range(len(colunas)):
    for j in range(i + 1, len(colunas)):
        if df[colunas[i]].equals(df[colunas[j]]):
            colunasIguais.append((colunas[i], colunas[j]))


# Drop primeira coluna dos pares de colunas iguais
colunasDuplicada = [primeiraColunas[0] for primeiraColunas in colunasIguais]
df = df.drop(columns=colunasDuplicada)
colunasIguais



[('qtde_contas_bancarias', 'qtde_contas_bancarias_especiais'),
 ('local_onde_reside', 'local_onde_trabalha')]

A partir do código é possível notar que existe mais um par de colunas iguais: o par ```('qtde_contas_bancarias', 'qtde_contas_bancarias_especiais')```. Assim como para o local onde reside/trabalha, uma das colunas foi removida. 

Uma coluna que foi decidida tirar também foi a coluna de ```meses_no_trabalho```, pois, conforme a descirção no dicionário, a grande maioria está com 0, mesmo com vínculo com uma empresa. Com isso, ficamos com o seguinte DF

In [16]:
df = df.drop(columns='meses_no_trabalho')
df.describe().T

,count,mean,std,min,25%,50%,75%,max
produto_solicitado,20000.0,1.284350,1.008239,1.0,1.0,1.0,1.0,7.0
dia_vencimento,20000.0,13.147250,6.748507,1.0,10.0,10.0,20.0,25.0
tipo_endereco,20000.0,1.006350,0.079435,1.0,1.0,1.0,1.0,2.0
idade,20000.0,42.352500,14.930177,7.0,31.0,40.0,52.0,106.0
estado_civil,20000.0,2.120850,1.332004,0.0,1.0,2.0,2.0,7.0
qtde_dependentes,20000.0,0.666400,1.236725,0.0,0.0,0.0,1.0,53.0
nacionalidade,20000.0,0.961600,0.202305,0.0,1.0,1.0,1.0,2.0
codigo_area_telefone_residencial,16466.0,64.362626,38.701585,1.0,25.0,69.0,100.0,126.0
tipo_residencia,19464.0,1.261303,0.883580,0.0,1.0,1.0,1.0,5.0
meses_na_residencia,18550.0,9.572453,10.649580,0.0,1.0,6.0,15.0,228.0


Partindo para a análise por linhas, o primeiro passo foi identificar os *outliers*. É possível identificar eles usando o percentil.

In [17]:
df.describe(percentiles=[0.01, 0.5, 0.99]).T

,count,mean,std,min,1%,50%,99%,max
produto_solicitado,20000.0,1.284350,1.008239,1.0,1.0,1.0,7.00,7.0
dia_vencimento,20000.0,13.147250,6.748507,1.0,1.0,10.0,25.00,25.0
tipo_endereco,20000.0,1.006350,0.079435,1.0,1.0,1.0,1.00,2.0
idade,20000.0,42.352500,14.930177,7.0,18.0,40.0,80.00,106.0
estado_civil,20000.0,2.120850,1.332004,0.0,1.0,2.0,7.00,7.0
qtde_dependentes,20000.0,0.666400,1.236725,0.0,0.0,0.0,5.00,53.0
nacionalidade,20000.0,0.961600,0.202305,0.0,0.0,1.0,1.00,2.0
codigo_area_telefone_residencial,16466.0,64.362626,38.701585,1.0,5.0,69.0,125.00,126.0
tipo_residencia,19464.0,1.261303,0.883580,0.0,0.0,1.0,5.00,5.0
meses_na_residencia,18550.0,9.572453,10.649580,0.0,0.0,6.0,45.00,228.0


Com a tabela acima é possível identificar algumas colunas com possíveis *outliers*, sendo elas:
- ```idade```: solicitante com 106 anos;
- ```qtde_dependentes```: solicitante com 53 dependentes;
- ```meses_na_residencia```: solicitante com 228 meses na residência;
- ```renda_mensal_regular```: solicitante com renda mensal de 959000 reais;
- ```renda_extra```: solicitante com renda extra de 194344 reais;
- ```valor_patrimonio_pessoal```: solicitante com patrimonio de 6000000 reais;

Apesar de serem alguns valores estranhos é possível sim ter valores como esse. Por exemplo, é possível alguem morar na mesma residência por 228 meses (19 anos). Por conta disso, foi tomada as seguintes decisões:

- ```idade```: É possível ter pessoas com essa idade, por conta disso mantive elas. A mudança feita na idade foi realizar um ```.clip(lower=18)``` pois a idade mínima para realizar solicitação é 18 anos; 
- ```qtde_dependentes```: Bem provavel que seja erro de digitação, pois o segundo maior é de 14 dependentes. Realizado ```.clip(upper=14)```;
- ```meses_na_residencia```: Apesar de ser alto, 228 meses na residência (19 anos) é algo possível. Nada foi feito;
- ```renda_mensal_regular```: Apesar de muito alto, outras pessoas possuem rendas tão altas quanto. Foi realizado a transformação ```log1p``` para deixar mais simétrico;
- ```renda_extra```: O segundo maior valor de renda extra é de 10200 reais, comparado aos 194344 é um salto muito grande. Nesse caso, o exemplo foi removido e foi aplicado a transformação ```log1p```;
- ```valor_patrimonio_pessoal```: Mesmo problema da coluna ```renda_extra```, o segundo maior valor é 10x menor que o maior valor. Exemplo também removido e aplicado a transformação ```log1p```.

Vale ressaltar que para realizar a analise foi utilizado o bloco abaixo para sempre ver os 20 maiores valores. Não coloquei a analise de todos pois seriam muitos blocos grandes printados.

In [18]:
df.nlargest(20, "valor_patrimonio_pessoal").T

,4406,17434,2562,7268,7466,9858,5908,8376,14302,18094,5222,7300,141,1160,1694,2146,3659,4379,5489,5572
produto_solicitado,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,1,2,1,1,2
dia_vencimento,10,25,1,10,10,20,10,25,5,25,25,10,20,20,15,5,15,5,15,20
forma_envio_solicitacao,internet,correio,presencial,internet,presencial,internet,internet,internet,internet,internet,internet,internet,internet,internet,internet,internet,internet,internet,internet,internet
tipo_endereco,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
sexo,F,F,F,M,M,M,F,F,F,F,M,F,F,M,F,F,M,M,F,M
idade,38,56,45,69,46,35,28,34,27,26,38,24,34,58,40,83,22,52,44,51
estado_civil,2,2,1,2,1,2,2,2,1,2,6,2,2,2,2,5,1,2,2,2
qtde_dependentes,1,0,1,0,0,1,1,5,0,2,0,1,1,0,3,0,0,2,0,0
nacionalidade,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
estado_onde_nasceu,RS,RJ,PR,SC,PR,PR,DF,RJ,GO,MT,PR,MT,SC,SC,SC,GO,SC,RS,SC,BA


In [19]:
df["idade"] = df["idade"].clip(lower=18)
df["qtde_dependentes"] = df["qtde_dependentes"].clip(upper=14)

idx_renda_extra = df["renda_extra"].idxmax()
idx_patrimonio = df["valor_patrimonio_pessoal"].idxmax()
df = df.drop(index=[idx_renda_extra, idx_patrimonio])

colunasFinanceiro = ["renda_mensal_regular", "renda_extra", "valor_patrimonio_pessoal"]
df[colunasFinanceiro] = np.log1p(df[colunasFinanceiro])

df.describe(percentiles=[0.01, 0.5, 0.99]).T

,count,mean,std,min,1%,50%,99%,max
produto_solicitado,19998.0,1.284378,1.008285,1.000000,1.000000,1.000000,7.000000,7.000000
dia_vencimento,19998.0,13.147565,6.748771,1.000000,1.000000,10.000000,25.000000,25.000000
tipo_endereco,19998.0,1.006301,0.079128,1.000000,1.000000,1.000000,1.000000,2.000000
idade,19998.0,42.352585,14.928495,18.000000,18.000000,40.000000,80.000000,106.000000
estado_civil,19998.0,2.120912,1.332047,0.000000,1.000000,2.000000,7.000000,7.000000
qtde_dependentes,19998.0,0.664416,1.183873,0.000000,0.000000,0.000000,5.000000,14.000000
nacionalidade,19998.0,0.961596,0.202315,0.000000,0.000000,1.000000,1.000000,2.000000
codigo_area_telefone_residencial,16465.0,64.363255,38.702676,1.000000,5.000000,69.000000,125.000000,126.000000
tipo_residencia,19462.0,1.261330,0.883621,0.000000,0.000000,1.000000,5.000000,5.000000
meses_na_residencia,18548.0,9.571706,10.649849,0.000000,0.000000,6.000000,45.000000,228.000000


# ANALISE DE NANS, MUITO CHATO DE DEPENDE DO MODELO QUE VAMOS ESCOLHER

Partindo para a análise por linhas, o primeiro passo foi procurar linhas com valores NaN para decidir o que seria feito com elas. Importante pontuar que  não tem linhas em branco por conta do argumento que usei para importar o csv para o dataframe.

In [20]:
tabelaAusentes = pd.DataFrame({
    "Valores preenchidos": df.notna().sum(),
    "NaN": df.isna().sum(),
    "% NaN": (df.isna().mean() * 100).round(1),
})


tabelaAusentes = tabelaAusentes[tabelaAusentes["NaN"] > 0].sort_values("NaN", ascending=False)
tabelaAusentes

,Valores preenchidos,NaN,% NaN
codigo_area_telefone_trabalho,5474,14524,72.6
estado_onde_trabalha,6426,13572,67.9
grau_instrucao_companheiro,7140,12858,64.3
profissao_companheiro,8486,11512,57.6
codigo_area_telefone_residencial,16465,3533,17.7
profissao,16901,3097,15.5
ocupacao,17021,2977,14.9
meses_na_residencia,18548,1450,7.3
estado_onde_nasceu,19176,822,4.1
tipo_residencia,19462,536,2.7


A primeira análise é relacionado as colunas ```codigo_area_telefone_trabalho``` e ```estado_onde_trabalha```, pois podem com NaN pela fato da pessoa não ter emprego. Para fazer essa análise foi utilizado a coluna ```vinculo_formal_com_empresa```.

In [21]:
pd.crosstab(df["vinculo_formal_com_empresa"], df["codigo_area_telefone_trabalho"].isna())

codigo_area_telefone_trabalho,False,True
vinculo_formal_com_empresa,,
N,397,10777
Y,5077,3747


In [22]:
pd.crosstab(df["vinculo_formal_com_empresa"], df["estado_onde_trabalha"].isna())

estado_onde_trabalha,False,True
vinculo_formal_com_empresa,,
N,346,10828
Y,6080,2744


# VER O QUE VAI FAZER AQUI 